### DEEP LEARNING

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow import keras
from keras import layers

import nltk


from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


In [2]:
# IMPORT AND PROCESSING
df = pd.read_csv('fake-reviews-dataset.csv')

df['num_characters'] = df['text_'].apply(len)


nltk.download('stopwords')
nltk.download('punkt')

nltk.download('punkt_tab')

# Imposta il percorso di ricerca dei dati NLTK
nltk.data.path.append('E:/nltk_data')  # Modifica con il percorso corretto, se diverso

# Prova a tokenizzare di nuovo
test_sentence = "Questo è un test di tokenizzazione."
tokens = nltk.word_tokenize(test_sentence)
print(tokens)

stop_words = set(stopwords.words('english'))


def preprocess_text(text):
    tokens = word_tokenize(text.lower())  # Tokenizza e trasforma tutto in minuscolo
    tokens = [word for word in tokens if word.isalpha()]  # Rimuove numeri e caratteri speciali
    tokens = [word for word in tokens if word not in stop_words]  # Rimuove stopwords
    return ' '.join(tokens)

df['cleaned_text'] = df['text_'].apply(preprocess_text)

display(df)


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ludov\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ludov\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\ludov\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


['Questo', 'è', 'un', 'test', 'di', 'tokenizzazione', '.']


,category,rating,label,text_,num_characters,cleaned_text
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor...",75,love well made sturdy comfortable love pretty
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I...",80,love great upgrade original mine couple years
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...,67,pillow saved back love look feel pillow
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i...",81,missing information use great product price
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...,85,nice set good quality set two months
...,...,...,...,...,...,...
40427,Clothing_Shoes_and_Jewelry_5,4.0,OR,I had read some reviews saying that this bra r...,1698,read reviews saying bra ran small ordered two ...
40428,Clothing_Shoes_and_Jewelry_5,5.0,CG,I wasn't sure exactly what it would be. It is ...,1305,sure exactly would little large small size thi...
40429,Clothing_Shoes_and_Jewelry_5,2.0,OR,"You can wear the hood by itself, wear it with ...",2007,wear hood wear hood wear jacket without hood s...
40430,Clothing_Shoes_and_Jewelry_5,1.0,CG,I liked nothing about this dress. The only rea...,1301,liked nothing dress reason gave stars ordered ...


In [3]:
# STEMMING
from nltk.stem import PorterStemmer

nltk.download('punkt')  # Solo la prima volta per scaricare il tokenizer

# Inizializza lo stemmer
stemmer = PorterStemmer()

# Esempio di testo da stemmare
text = "running runner ran easily the boys are playing games"

# Tokenizza il testo
words = word_tokenize(text)

# Applica lo stemming su ogni parola
stemmed_words = [stemmer.stem(word) for word in words]

# Stampa le parole stemmate
print("Parole originali:", words)
print("Parole stemmate:", stemmed_words)

# Funzione per applicare lo stemming
def apply_stemming(text):
    # Tokenizza il testo
    words = word_tokenize(text)
    # Applica lo stemming su ogni parola
    stemmed_words = [stemmer.stem(word) for word in words]
    # Ritorna il testo stemmato come stringa
    return ' '.join(stemmed_words)

# Applica la funzione di stemming alla colonna 'cleaned_text'
df['stemmed_text'] = df['cleaned_text'].apply(apply_stemming)

# Stampa il DataFrame con la nuova colonna stemmata
display(df.head(6))
if 'cleaned_text' not in df.columns:
    print("Colonna 'cleaned_text' non trovata nel DataFrame.")

if 'stemmed_text' not in df.columns:
    print("Colonna 'stemmed_text' non trovata nel DataFrame.")


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\ludov\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Parole originali: ['running', 'runner', 'ran', 'easily', 'the', 'boys', 'are', 'playing', 'games']
Parole stemmate: ['run', 'runner', 'ran', 'easili', 'the', 'boy', 'are', 'play', 'game']


,category,rating,label,text_,num_characters,cleaned_text,stemmed_text
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor...",75,love well made sturdy comfortable love pretty,love well made sturdi comfort love pretti
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I...",80,love great upgrade original mine couple years,love great upgrad origin mine coupl year
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...,67,pillow saved back love look feel pillow,pillow save back love look feel pillow
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i...",81,missing information use great product price,miss inform use great product price
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...,85,nice set good quality set two months,nice set good qualiti set two month
5,Home_and_Kitchen_5,3.0,CG,I WANTED DIFFERENT FLAVORS BUT THEY ARE NOT.,44,wanted different flavors,want differ flavor


In [4]:
df.columns = df.columns.str.strip()  # Rimuovi spazi bianchi dai nomi delle colonne
print(df.columns.duplicated())
df = df.dropna(subset=['stemmed_text'])


[False False False False False False False]


In [5]:
# BAG OF WORDS
from sklearn.feature_extraction.text import TfidfVectorizer

df = df.dropna(subset=['stemmed_text'])
df['stemmed_text'] = df['stemmed_text'].fillna('')

vectorizer = TfidfVectorizer(max_features=5000)     #massimo 5000 parole piu importanti
X = vectorizer.fit_transform(df['stemmed_text'])    #trasforma i testi in vettori TF-IDF

In [6]:
# SENTIMENT
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score


# Preprocessing
stop_words = set(stopwords.words('english'))
def preprocess_text(text):
    tokens = nltk.word_tokenize(text.lower())
    tokens = [word for word in tokens if word.isalnum() and word not in stop_words]
    return ' '.join(tokens)

df['processed_text'] = df['text_'].apply(preprocess_text)

from textblob import TextBlob
#assegno a ciascuna recensione il sentiment effettivo
def get_true_sentiment(rating):
    if rating <= 2:
        return "Negative"
    elif rating == 3:
        return "Neutral"
    elif rating >= 4:
        return "Positive"
    
# Funzione per determinare il sentiment label basato sulla polarità
def get_sentiment_label(polarity):
    if polarity > 0:
        return "Positive"
    if polarity < 0:
        return "Negative"
    else:
        return "Neutral"
    
df['true_sentiment'] = df['rating'].apply(get_true_sentiment)

# Aggiunge la colonna 'sentiment' solo se non esiste già
if 'sentiment' not in df.columns:
    sentiments = []
    for text in df['text_']:
        # Calcola il sentiment usando TextBlob
        sentiment = TextBlob(text).sentiment
        polarity = sentiment.polarity
        # Determina il label usando la polarità
        sentiment_label = get_sentiment_label(polarity)
        sentiments.append(sentiment_label)

 # Aggiunge la lista dei sentimenti calcolati al DataFrame
    df['sentiment'] = sentiments

df['match'] = (df['true_sentiment'] == df['sentiment']).astype(int)

## DEEP LEARNING

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Prepara le feature e le etichette
X = df.drop('label', axis=1)  # 'label' è la colonna che indica se è vero/falso
y = df['label'].apply(lambda x: 1 if x == 'valore_positivo' else 0)  # Converte le etichette in 0 o 1

# Codifica le colonne testuali nelle feature
label_encoders = {}
for column in X.select_dtypes(include=['object']).columns:
    le = LabelEncoder()
    X[column] = le.fit_transform(X[column])
    label_encoders[column] = le

# Dividi i dati in set di allenamento e di test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Dividi X_train per il validation set
X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.25, random_state=42)  # 0.25 * 0.8 = 0.2 del totale

# Standardizza le feature
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# Stampa il DataFrame finale
display(df)


,category,rating,label,text_,num_characters,cleaned_text,stemmed_text,processed_text,true_sentiment,sentiment,match
0,Home_and_Kitchen_5,5.0,CG,"Love this! Well made, sturdy, and very comfor...",75,love well made sturdy comfortable love pretty,love well made sturdi comfort love pretti,love well made sturdy comfortable love pretty,Positive,Positive,1
1,Home_and_Kitchen_5,5.0,CG,"love it, a great upgrade from the original. I...",80,love great upgrade original mine couple years,love great upgrad origin mine coupl year,love great upgrade original mine couple years,Positive,Positive,1
2,Home_and_Kitchen_5,5.0,CG,This pillow saved my back. I love the look and...,67,pillow saved back love look feel pillow,pillow save back love look feel pillow,pillow saved back love look feel pillow,Positive,Positive,1
3,Home_and_Kitchen_5,1.0,CG,"Missing information on how to use it, but it i...",81,missing information use great product price,miss inform use great product price,missing information use great product price,Negative,Positive,0
4,Home_and_Kitchen_5,5.0,CG,Very nice set. Good quality. We have had the s...,85,nice set good quality set two months,nice set good qualiti set two month,nice set good quality set two months,Positive,Positive,1
...,...,...,...,...,...,...,...,...,...,...,...
40427,Clothing_Shoes_and_Jewelry_5,4.0,OR,I had read some reviews saying that this bra r...,1698,read reviews saying bra ran small ordered two ...,read review say bra ran small order two band c...,read reviews saying bra ran small ordered two ...,Positive,Positive,1
40428,Clothing_Shoes_and_Jewelry_5,5.0,CG,I wasn't sure exactly what it would be. It is ...,1305,sure exactly would little large small size thi...,sure exactli would littl larg small size think...,sure exactly would little large small size thi...,Positive,Positive,1
40429,Clothing_Shoes_and_Jewelry_5,2.0,OR,"You can wear the hood by itself, wear it with ...",2007,wear hood wear hood wear jacket without hood s...,wear hood wear hood wear jacket without hood s...,wear hood wear hood wear jacket without hood s...,Negative,Positive,0
40430,Clothing_Shoes_and_Jewelry_5,1.0,CG,I liked nothing about this dress. The only rea...,1301,liked nothing dress reason gave stars ordered ...,like noth dress reason gave star order size fi...,liked nothing dress reason gave 4 stars ordere...,Negative,Positive,0


In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, Embedding, Bidirectional, BatchNormalization
from keras.callbacks import EarlyStopping


# Creazione del modello
model = Sequential()
model.add(Embedding(input_dim=100000, output_dim=128, input_length=200))
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.6))
model.add(BatchNormalization())
model.add(LSTM(32))
model.add(Dropout(0.6))
model.add(Dense(1, activation='sigmoid'))


c:\Users\ludov\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [9]:
# compila il modello
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)


In [10]:
# addestramento
history = model.fit(X_train_scaled, y_train, epochs=10, validation_data=(X_val, y_val))

Epoch 1/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 126s 156ms/step - accuracy: 0.9876 - loss: 0.0463 - val_accuracy: 1.0000 - val_loss: 7.0533e-05
Epoch 2/10
759/759 ━━━━━━━━━━━━━━━━━━━━ 118s 155ms/step - accuracy: 1.0000 - loss: 3.4738e-04 - val_accuracy: 1.0000 - val_loss: 1.5571e-05
Epoch 3/10
424/759 ━━━━━━━━━━━━━━━━━━━━ 58s 175ms/step - accuracy: 1.0000 - loss: 1.6884e-04

KeyboardInterrupt: 

In [11]:
# valutazione del modello
test_loss, test_acc = model.evaluate(X_test_scaled, y_test)
print(f"Test Accuracy: {test_acc:.2f}")


253/253 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 1.0000 - loss: 4.9317e-06
Test Accuracy: 1.00


In [12]:
# visualizzazione dei risultati, per vedere come si è comportato il modello
import matplotlib.pyplot as plt

# plot dell'accuracy
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

# plot della loss
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend()
plt.show()


NameError: name 'history' is not defined

In [1]:
model.summary()

NameError: name 'model' is not defined